# Using Apache Spark with Delta Lake

Every table written in this course so far has been a Delta table, because
unless otherwise specified, all tables on Databricks are Delta Lake tables
([What is Delta Lake?](https://docs.databricks.com/aws/en/delta/)).
{doc}`Reading and Writing Data <../01_introduction_to_apache_spark/02_spark_dataframes_and_sql/02_reading_and_writing_data>`
introduced it as "Parquet files plus a transaction log". This notebook
opens up that transaction log and uses the features it makes possible:
updating and merging data in place, enforcing a schema and constraints,
reading old versions of a table, tracking changes, and keeping tables
healthy with maintenance commands.

In [1]:
from pyspark.sql import functions as F

from project.utils.sparksession import get_spark_session

spark = get_spark_session(profile="free-edition")

SALES = "workspace.default.delta02_sales"
for table in ["delta02_sales", "delta02_small_files"]:
    spark.sql(f"DROP TABLE IF EXISTS workspace.default.{table}")

Create a Delta table from the bakehouse transactions, without the card
numbers:

In [2]:
(
    spark.read.table("samples.bakehouse.sales_transactions")
    .drop("cardNumber")
    .write.saveAsTable(SALES)
)

spark.read.table(SALES).count()

3333

## The Transaction Log

Delta Lake extends Parquet data files with a file-based transaction log
([What is Delta Lake?](https://docs.databricks.com/aws/en/delta/)). A write
first writes its data files, then commits an entry to the transaction log
that lists them. Each commit increments the table version and makes the new
files visible, and the current state of the table is all the data files the
log marks as valid
([ACID guarantees on Databricks](https://docs.databricks.com/aws/en/lakehouse/acid#how-does-databricks-implement-atomicity)).

This design gives Delta tables the properties that plain Parquet folders
lack
([ACID guarantees on Databricks](https://docs.databricks.com/aws/en/lakehouse/acid)):

- **Atomic commits.** A transaction either succeeds or fails completely;
  files from a failed write never become part of the table.
- **Isolation.** All reads use snapshot isolation: a reader sees one
  specific version of the table, even while others are writing.
- **History.** Each write is recorded with its operation, user and
  timestamp, and older versions can still be read
  ([Work with table history](https://docs.databricks.com/aws/en/delta/history)).

`DESCRIBE DETAIL` summarises the current version of a table:

In [3]:
spark.sql(f"DESCRIBE DETAIL {SALES}").select(
    "format", "numFiles", "sizeInBytes", "clusteringColumns", "tableFeatures"
).show(truncate=False)

+------+--------+-----------+-----------------+-----------------------------------------+
|format|numFiles|sizeInBytes|clusteringColumns|tableFeatures                            |
+------+--------+-----------+-----------------+-----------------------------------------+
|delta |1       |35011      |[]               |[appendOnly, deletionVectors, invariants]|
+------+--------+-----------+-----------------+-----------------------------------------+



`DESCRIBE HISTORY` lists the commits, newest first. So far there is only
the one that created the table:

In [4]:
spark.sql(f"DESCRIBE HISTORY {SALES}").select("version", "timestamp", "operation", "operationMetrics").show(truncate=60)

+-------+-------------------+----------------------+------------------------------------------------------------+
|version|          timestamp|             operation|                                            operationMetrics|
+-------+-------------------+----------------------+------------------------------------------------------------+
|      0|2026-10-12 21:06:44|CREATE TABLE AS SELECT|{numFiles -> 1, numOutputRows -> 3333, numOutputBytes -> ...|
+-------+-------------------+----------------------+------------------------------------------------------------+



## Changing Data in Place

Plain Parquet files cannot be changed: to fix one row you rewrite the whole
dataset. Delta tables support `UPDATE`, `DELETE` and `MERGE`. They work
by writing new files and committing a change that swaps them for the old
ones. With **deletion vectors**, a delete or update doesn't even rewrite the
affected files: it marks the changed rows in metadata, applies that at read
time, and leaves the physical rewrite to a later `OPTIMIZE`
([Deletion vectors](https://docs.databricks.com/aws/en/delta/deletion-vectors)).
Deletion vectors are a table feature; whether new tables get them depends on
a workspace setting. In the course's workspace they do: the `tableFeatures`
column of the `DESCRIBE DETAIL` output above lists `deletionVectors`.

Suppose the price of Pearly Pies was recorded wrongly and must be raised to
4, and all transactions of one customer must be removed at their request:

In [5]:
spark.sql(f"""
    UPDATE {SALES}
    SET unitPrice = 4, totalPrice = quantity * 4
    WHERE product = 'Pearly Pies'
""").show()

+-----------------+
|num_affected_rows|
+-----------------+
|              550|
+-----------------+



In [6]:
spark.sql(f"DELETE FROM {SALES} WHERE customerID = 2000112").show()

+-----------------+
|num_affected_rows|
+-----------------+
|               20|
+-----------------+



Each statement returns the number of rows it affected, and each is a new
commit in the history.

### Upserts with `MERGE`

The most important write operation in data engineering is the **upsert**:
update the rows that already exist, insert the ones that do not. `MERGE`
does both, from a source table, view or DataFrame into a target Delta table
([Upsert into a Delta Lake table using merge](https://docs.databricks.com/aws/en/delta/merge)),
in one atomic commit. Here a batch of corrections arrives: one
transaction with a corrected quantity, and one transaction that was
missing:

In [7]:
corrections = spark.createDataFrame(
    [
        (1002961, 2000253, 3000047, "2024-05-14 12:17:01", "Golden Gate Ginger", 10, 3, 30, "amex"),
        (1009999, 2000001, 3000001, "2024-05-17 09:30:00", "Tokyo Tidbits", 2, 3, 6, "visa"),
    ],
    schema="""
        transactionID BIGINT, customerID BIGINT, franchiseID BIGINT, dateTime STRING,
        product STRING, quantity BIGINT, unitPrice BIGINT, totalPrice BIGINT, paymentMethod STRING
    """,
).withColumn("dateTime", F.to_timestamp("dateTime"))

corrections.createOrReplaceTempView("corrections")

spark.sql(f"""
    MERGE INTO {SALES} AS target
    USING corrections AS source
    ON target.transactionID = source.transactionID
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""").show()

+-----------------+----------------+----------------+-----------------+
|num_affected_rows|num_updated_rows|num_deleted_rows|num_inserted_rows|
+-----------------+----------------+----------------+-----------------+
|                2|               1|               0|                1|
+-----------------+----------------+----------------+-----------------+



The same operation is available from Python through the `DeltaTable` API,
which is convenient inside functions:

```python
from delta.tables import DeltaTable

(
    DeltaTable.forName(spark, SALES).alias("target")
    .merge(corrections.alias("source"), "target.transactionID = source.transactionID")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)
```

:::::{dropdown} Example: What Does the `MERGE` Do to Each Row?
:open:
:color: primary
:icon: pencil
:class-body: example-body

The source holds two transactions. For each, `MERGE` looks for a target row
with the same `transactionID`.

**Transaction 1002961 exists in the target**, with a quantity of 8 and a
total of 24. The `WHEN MATCHED` clause replaces it with the source row: a
quantity of 10 and a total of $10 \times 3 = 30$.

**Transaction 1009999 does not exist in the target.** The `WHEN NOT MATCHED`
clause inserts it.

The result is 1 updated row and 1 inserted row, as the output above shows.
Running the same `MERGE` again changes nothing, because both rows now match
and are updated with identical values. That makes a keyed `MERGE`
**idempotent**: safe to retry, as described in
{doc}`ETL with the DataFrame API <../01_introduction_to_apache_spark/04_etl_with_the_dataframe_api/index>`.
:::::

Checking the corrected row:

In [8]:
spark.read.table(SALES).filter(F.col("transactionID").isin(1002961, 1009999)).select(
    "transactionID", "product", "quantity", "totalPrice"
).show()

+-------------+------------------+--------+----------+
|transactionID|           product|quantity|totalPrice|
+-------------+------------------+--------+----------+
|      1002961|Golden Gate Ginger|      10|        30|
|      1009999|     Tokyo Tidbits|       2|         6|
+-------------+------------------+--------+----------+



## Schema Enforcement and Evolution

A Delta table has a schema, and Databricks enforces it on write
([Schema enforcement](https://docs.databricks.com/aws/en/delta/schema-validation)).
Appending data with an extra column or a misspelt name fails instead of
silently corrupting the table:

In [9]:
with_channel = spark.read.table(SALES).limit(3).withColumn("channel", F.lit("online"))

with_channel.write.mode("append").saveAsTable(SALES)

AnalysisException: [DELTA_METADATA_MISMATCH] A metadata mismatch was detected when writing to the Delta table.
- A schema mismatch detected when writing to the Delta table (Table ID: 3506014c-d2a8-47db-9854-76658003a1ed).
To enable schema migration using DataFrameWriter or DataStreamWriter, please set: '.option("mergeSchema", "true")'.
For other operations, set the session configuration spark.databricks.delta.schema.autoMerge.enabled to "true". See the documentation specific to the operation for details.

Table schema:
root
 |-- transactionID: long (nullable = true)
 |-- customerID: long (nullable = true)
 |-- franchiseID: long (nullable = true)
 |-- dateTime: timestamp (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- unitPrice: long (nullable = true)
 |-- totalPrice: long (nullable = true)
 |-- paymentMethod: string (nullable = true)


Data schema:
root
 |-- transactionID: long (nullable = true)
 |-- customerID: long (nullable = true)

When the new column is intended, allow the schema to **evolve** with the
`mergeSchema` option
([Schema enforcement](https://docs.databricks.com/aws/en/delta/schema-validation#modify-a-table-schema)).
Existing rows get `NULL` in the new column:

In [10]:
with_channel.write.mode("append").option("mergeSchema", "true").saveAsTable(SALES)

spark.read.table(SALES).groupBy("channel").count().show()

+-------+-----+
|channel|count|
+-------+-----+
|   NULL| 3314|
| online|    3|
+-------+-----+



### Constraints

Schema enforcement checks columns and types, not values. **Constraints**
check values. Adding a `CHECK` constraint verifies that all existing rows
satisfy it, and a later write that violates it fails the whole transaction
([Constraints on Databricks](https://docs.databricks.com/aws/en/tables/constraints#check-constraint)):

In [11]:
spark.sql(f"ALTER TABLE {SALES} ADD CONSTRAINT positive_quantity CHECK (quantity > 0)")

bad_row = spark.read.table(SALES).limit(1).withColumn("quantity", F.lit(-5).cast("bigint"))

In [12]:
bad_row.write.mode("append").saveAsTable(SALES)

UnknownException: (com.databricks.sql.transaction.tahoe.schema.DeltaInvariantViolationException) [DELTA_VIOLATE_CONSTRAINT_WITH_VALUES] CHECK constraint positive_quantity (quantity > 0) violated by row with values:
 - quantity : -5.

JVM stacktrace:
com.databricks.sql.transaction.tahoe.schema.DeltaInvariantViolationException
	at com.databricks.sql.transaction.tahoe.schema.DeltaInvariantViolationException$.getConstraintViolationWithValuesException(InvariantViolationException.scala:82)
	at com.databricks.photon.PhotonException$.getSparkException(PhotonException.scala:550)
	at com.databricks.photon.PhotonWriteResultHandler.getResult(PhotonWriteStageExec.scala:153)
	at com.databricks.photon.PhotonBasicEvaluatorFactory$PhotonBasicEvaluator$$anon$1.open(PhotonBasicEvaluatorFactory.scala:290)
	at com.databricks.photon.PhotonBasicEvaluatorFactory$PhotonBasicEvaluator$$anon$1.hasNextImpl(PhotonBasicEvaluatorFactory.scala:295)
	at com.databricks.photon.PhotonBasicEvaluatorFactory$PhotonBasicEval

The whole write is rejected, so a table protected by constraints never
contains a mix of good and bad rows from the same batch. `NOT NULL`
constraints are enforced the same way, and added with
`ALTER TABLE ... ALTER COLUMN ... SET NOT NULL`
([Constraints on Databricks](https://docs.databricks.com/aws/en/tables/constraints#not-null-constraint)).

## Time Travel

Because old commits stay in the log, you can query the table **as it was**
at an earlier version or time. The history now has a commit for each
change made above:

In [13]:
history = spark.sql(f"DESCRIBE HISTORY {SALES}").select("version", "operation", "operationMetrics.numOutputRows")
history.orderBy("version").show(truncate=False)

+-------+----------------------+-------------+
|version|operation             |numOutputRows|
+-------+----------------------+-------------+
|0      |CREATE TABLE AS SELECT|3333         |
|1      |UPDATE                |NULL         |
|2      |OPTIMIZE              |NULL         |
|3      |DELETE                |NULL         |
|4      |MERGE                 |2            |
|5      |WRITE                 |3            |
|6      |ADD CONSTRAINT        |NULL         |
+-------+----------------------+-------------+



Databricks may also add commits of its own, such as an `OPTIMIZE` that
compacts files in the background, so version numbers are not always the
ones you would predict. Read them from the history.

Read version 0, the table as first created, with `versionAsOf`, or with
`VERSION AS OF` in SQL. `timestampAsOf` and `TIMESTAMP AS OF` select a
version by time instead
([Work with table history](https://docs.databricks.com/aws/en/delta/history#time-travel-syntax)):

In [14]:
original = spark.read.option("versionAsOf", 0).table(SALES)
current = spark.read.table(SALES)

original.count(), current.count()

(3333, 3317)

Comparing two versions is an easy way to audit what changed, for example
the Pearly Pies totals before and after the price correction:

In [15]:
def pearly_pies_revenue(df):
    return df.filter(F.col("product") == "Pearly Pies").agg(F.sum("totalPrice")).first()[0]


pearly_pies_revenue(original), pearly_pies_revenue(current)

(10785, 14712)

The revenue rose with the price, though not by exactly 4/3, because the
deleted customer's sales and the three rows appended in the schema evolution
example changed the total too.

If a change was a mistake, `RESTORE` rolls the table back to an earlier
version or timestamp
([RESTORE](https://docs.databricks.com/aws/en/sql/language-manual/delta-restore)).
It does not erase history: the restore is itself a new commit, as the last
line of the history below shows, so you can even undo the undo.

In [16]:
spark.sql(f"RESTORE TABLE {SALES} TO VERSION AS OF 0").select("num_restored_files", "num_removed_files").show()

print(spark.read.table(SALES).count())
spark.sql(f"DESCRIBE HISTORY {SALES}").select("version", "operation").orderBy(F.desc("version")).show(2)

+------------------+-----------------+
|num_restored_files|num_removed_files|
+------------------+-----------------+
|                 1|                4|
+------------------+-----------------+



3333


+-------+--------------+
|version|     operation|
+-------+--------------+
|      7|       RESTORE|
|      6|ADD CONSTRAINT|
+-------+--------------+
only showing top 2 rows


Time travel only reaches back as far as the data files still exist.
`VACUUM`, covered below, deletes files that are no longer referenced once
they are older than the retention threshold, 7 days by default, and with
them the ability to query those versions
([Remove unused data files with vacuum](https://docs.databricks.com/aws/en/delta/vacuum#caveats-for-vacuum)).

## Change Data Feed

Downstream tables often need to know **what changed** in a table, not just
its current state: which rows were inserted, updated or deleted, so they can
apply the same changes. With the **change data feed** enabled through the
`delta.enableChangeDataFeed` table property, Delta records change events for
all data written to the table from then on; batch reads need a starting
version
([Use Delta Lake change data feed](https://docs.databricks.com/aws/en/delta/delta-change-data-feed)):

In [17]:
spark.sql(f"ALTER TABLE {SALES} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

cdf_start = spark.sql(f"DESCRIBE HISTORY {SALES}").agg(F.max("version")).first()[0] + 1

spark.sql(f"UPDATE {SALES} SET quantity = quantity + 1 WHERE transactionID = 1002961")
spark.sql(f"DELETE FROM {SALES} WHERE transactionID = 1003007")

(
    spark.read.option("readChangeFeed", "true").option("startingVersion", cdf_start).table(SALES)
    .select("transactionID", "quantity", "_change_type", "_commit_version")
    .orderBy("_commit_version", "transactionID", "_change_type")
    .show()
)

+-------------+--------+----------------+---------------+
|transactionID|quantity|    _change_type|_commit_version|
+-------------+--------+----------------+---------------+
|      1002961|       9|update_postimage|              9|
|      1002961|       8| update_preimage|              9|
|      1003007|      36|          delete|             10|
+-------------+--------+----------------+---------------+



An update produces two rows, `update_preimage` and `update_postimage`, and a
delete produces one
([Use Delta Lake change data feed](https://docs.databricks.com/aws/en/delta/delta-change-data-feed)).
The change feed can also be read as a stream, which Databricks recommends, which is
how incremental pipelines propagate updates and deletes from one table to
the next, as in
{doc}`Spark Structured Streaming <../03_stream_processing_and_analysis/02_spark_structured_streaming>`.

## Table Maintenance

### Compacting Small Files with `OPTIMIZE`

Many small writes, such as a stream committing every minute or a job that
writes with too many partitions, leave a table with many small files.
Every file costs a request to cloud storage and some metadata work, so a
table of thousands of tiny files reads far more slowly than the same data in
a few large files. To see the effect, write the transactions as 200 small
files:

In [18]:
SMALL = "workspace.default.delta02_small_files"

spark.read.table("samples.bakehouse.sales_transactions").repartition(200).write.saveAsTable(SMALL)

spark.sql(f"DESCRIBE DETAIL {SMALL}").select("numFiles", "sizeInBytes").show()

+--------+-----------+
|numFiles|sizeInBytes|
+--------+-----------+
|     200|     732872|
+--------+-----------+



`OPTIMIZE` rewrites data files into a better layout, in a new commit. It
makes no data changes, readers are not interrupted, and a second run on the
same data has no effect
([Optimize data file layout](https://docs.databricks.com/aws/en/delta/optimize)):

In [19]:
spark.sql(f"OPTIMIZE {SMALL}").select("metrics.numFilesRemoved", "metrics.numFilesAdded").show()

spark.sql(f"DESCRIBE DETAIL {SMALL}").select("numFiles", "sizeInBytes").show()

+---------------+-------------+
|numFilesRemoved|numFilesAdded|
+---------------+-------------+
|            200|            1|
+---------------+-------------+



+--------+-----------+
|numFiles|sizeInBytes|
+--------+-----------+
|       1|      56750|
+--------+-----------+



The data is unchanged. Only its layout is better.

### Removing Old Files with `VACUUM`

`OPTIMIZE`, `UPDATE`, `DELETE` and `MERGE` all leave old files behind,
because earlier versions of the table still refer to them. `VACUUM` removes
data files that are no longer referenced by the table and are older than the
retention threshold, 7 days by default; the threshold is the table property
`delta.deletedFileRetentionDuration`
([Remove unused data files with vacuum](https://docs.databricks.com/aws/en/delta/vacuum)):

```sql
VACUUM workspace.default.delta02_sales              -- remove unreferenced files older than the threshold
VACUUM workspace.default.delta02_sales DRY RUN      -- list the files it would remove

ALTER TABLE workspace.default.delta02_sales
  SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = 'interval 14 days')
```

Databricks strongly recommends a retention interval of at least 7 days
([VACUUM](https://docs.databricks.com/aws/en/sql/language-manual/delta-vacuum)):
with a shorter one, `VACUUM` can delete files that a long-running job still
needs
([Remove unused data files with vacuum](https://docs.databricks.com/aws/en/delta/vacuum#vacuum-and-low-retention-thresholds)).

### Predictive Optimization

On Unity Catalog managed tables, Databricks can run this maintenance for
you. **Predictive optimization** automatically runs `OPTIMIZE`, `VACUUM` and
`ANALYZE` on Unity Catalog managed tables, and is enabled by default for
accounts created on or after November 11, 2024. `DESCRIBE TABLE EXTENDED`
shows whether it is on
([Predictive optimization](https://docs.databricks.com/aws/en/optimizations/predictive-optimization)).
In the course's workspace it is enabled, inherited from the metastore:

In [20]:
spark.sql(f"DESCRIBE EXTENDED {SALES}").filter(F.col("col_name") == "Predictive Optimization").show(truncate=False)

+-----------------------+---------------------------------------------------------+-------+
|col_name               |data_type                                                |comment|
+-----------------------+---------------------------------------------------------+-------+
|Predictive Optimization|ENABLE (inherited from METASTORE metastore_aws_us_east_2)|       |
+-----------------------+---------------------------------------------------------+-------+



The operations it has run are recorded in the system table
`system.storage.predictive_optimization_operations_history`
([Predictive optimization](https://docs.databricks.com/aws/en/optimizations/predictive-optimization#track-predictive-optimization-with-system-tables)).

## Liquid Clustering

Delta tables collect statistics for each data file on write, and queries use
them to skip irrelevant files
([Data skipping](https://docs.databricks.com/aws/en/delta/data-skipping)).
That only works well if rows with similar values are stored together.
**Liquid clustering** arranges the data that way for the clustering keys,
and Databricks recommends it for all new tables, replacing partitioning and
`ZORDER`. `CLUSTER BY AUTO` lets Databricks choose the keys, and `OPTIMIZE`
(run by predictive optimization on managed tables) clusters the data
incrementally
([Use liquid clustering for tables](https://docs.databricks.com/aws/en/tables/clustering)):

```sql
CREATE TABLE sales_by_day CLUSTER BY (sale_date) AS SELECT ...
ALTER TABLE workspace.default.delta02_sales CLUSTER BY (franchiseID)
ALTER TABLE workspace.default.delta02_sales CLUSTER BY AUTO   -- let Databricks choose the columns
```

How much it speeds up queries is measured in
{doc}`Optimizing Apache Spark <03_optimizing_apache_spark>`.

Unlike partitioning, clustering keys can be redefined without rewriting
existing data: later `OPTIMIZE` and write operations use the new keys, and
`OPTIMIZE ... FULL` reclusters the existing data when you want it to
([Use liquid clustering for tables](https://docs.databricks.com/aws/en/tables/clustering#change-clustering-keys)).

## Choosing a Write or Maintenance Operation

| Need | Use |
| --- | --- |
| Insert new rows and update existing ones | `MERGE` |
| Fix values in place | `UPDATE` |
| Remove rows, for example on a deletion request | `DELETE`, then let `VACUUM` remove the old files after the retention period |
| Add a column on purpose | `mergeSchema` on the write |
| Guard values | `CHECK` and `NOT NULL` constraints |
| Audit or roll back a change | `DESCRIBE HISTORY`, time travel, `RESTORE` |
| Propagate row-level changes downstream | Change data feed |
| Compact files and remove old ones | Predictive optimization on managed tables; `OPTIMIZE` and `VACUUM` by hand otherwise |

## Common Mistakes

- **Expecting `DELETE` to remove data from storage immediately.** Older
  versions still reference the files until `VACUUM` removes them after the
  retention threshold
  ([Remove unused data files with vacuum](https://docs.databricks.com/aws/en/delta/vacuum)).
- **Shortening the `VACUUM` retention below 7 days.** Long-running jobs can
  lose files they still need, and time travel to older versions is lost
  ([VACUUM](https://docs.databricks.com/aws/en/sql/language-manual/delta-vacuum)).
- **Turning on the change data feed and expecting earlier changes.** It
  only records changes made after it was enabled
  ([Use Delta Lake change data feed](https://docs.databricks.com/aws/en/delta/delta-change-data-feed)).
- **Using `mergeSchema` by default.** It turns schema mistakes into new
  columns; enable it only when the change is intended.
- **Merging a source with duplicate keys.** Only one source row may match
  each target row
  ([Upsert into a Delta Lake table using merge](https://docs.databricks.com/aws/en/delta/merge#merge-operation-semantics)).
- **Hard-coding version numbers.** Databricks can add its own commits, such
  as background `OPTIMIZE`; read versions from `DESCRIBE HISTORY`.

## Cleaning Up

In [21]:
for table in ["delta02_sales", "delta02_small_files"]:
    spark.sql(f"DROP TABLE IF EXISTS workspace.default.{table}");

## Resources

- [What is Delta Lake? (Databricks)](https://docs.databricks.com/aws/en/delta/)
- [Upsert into a Delta Lake table using merge (Databricks)](https://docs.databricks.com/aws/en/delta/merge)
- [Work with Delta Lake table history (Databricks)](https://docs.databricks.com/aws/en/delta/history)
- [Use Delta Lake change data feed (Databricks)](https://docs.databricks.com/aws/en/delta/delta-change-data-feed)
- [Predictive optimization for Unity Catalog managed tables (Databricks)](https://docs.databricks.com/aws/en/optimizations/predictive-optimization)
- [Use liquid clustering for tables (Databricks)](https://docs.databricks.com/aws/en/delta/clustering)